# M7 Manual Agent Loop - Competing Notebook

Build a ReAct-style agent loop by hand in pure Python.

Core rule for this notebook:

```text
The model requests. Python executes.
```

Final task:

> Can Alice add more AAPL under the guideline? Show the facts you checked.

## 0. Setup

The notebook defaults to deterministic planning so it always runs in class. Later, an optional local SmolLM2 planner can replace the deterministic planner.

In [ ]:
import json
from pathlib import Path
from time import perf_counter

from pydantic import BaseModel, Field

USE_LOCAL_MODEL = False

## 1. Smallest Agent Loop

First, remove the model. An agent loop is ordinary control flow:

```text
perception -> planning -> tool execution -> observation -> planning
```

In [ ]:
state = {
    "question": "Can Alice add more AAPL under the guideline?",
    "observations": [],
}

def first_planner(state: dict) -> dict:
    if not state["observations"]:
        return {"tool": "get_current_price", "args": {"symbol": "AAPL"}}
    return {"final": f"Checked facts: {state['observations']}"}

def get_current_price(symbol: str) -> dict:
    prices = {"AAPL": 108.0, "MSFT": 196.0}
    if symbol not in prices:
        raise ValueError(f"Unknown symbol: {symbol}")
    return {"symbol": symbol, "price": prices[symbol]}

for turn in range(3):
    step = first_planner(state)
    print("PLAN:", step)

    if "final" in step:
        print("FINAL:", step["final"])
        break

    # Actual tool execution happens here.
    result = get_current_price(**step["args"])
    observation = {"tool": step["tool"], "result": result}
    state["observations"].append(observation)
    print("OBSERVE:", observation)

Concept-to-code mapping:

```text
state = perception + observations
first_planner(state) = planning
get_current_price(...) = tool execution
state['observations'].append(...) = feedback
```

## 2. Tool Request As JSON

Function calling is not magic. The planner emits structured text. Python parses, validates, and executes the actual function call.

In [ ]:
class CurrentPriceArgs(BaseModel):
    symbol: str = Field(description="Ticker symbol, for example AAPL")

raw_step = '{"tool":"get_current_price","args":{"symbol":"AAPL"}}'
print("MODEL OUTPUT:", raw_step)

step = json.loads(raw_step)
args = CurrentPriceArgs.model_validate(step["args"])
print("VALIDATED:", step["tool"], args.model_dump())

# Actual tool execution happens here.
result = get_current_price(**args.model_dump())
observation = {"tool": step["tool"], "result": result}
print("OBSERVATION:", observation)

## 3. Optional Local Model Planner

This cell defines the model injection point. Keep `USE_LOCAL_MODEL = False` for deterministic classroom execution. Set it to `True` only if the local SmolLM2 model and dependencies are available.

In [ ]:
def find_model_path() -> Path:
    candidates = [
        Path("../OFFLINE-AI-Models/smollm2-135m-instruct"),
        Path("OFFLINE-AI-Models/smollm2-135m-instruct"),
        Path("../../OFFLINE-AI-Models/smollm2-135m-instruct"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("smollm2-135m-instruct")

def ask_local_model(prompt: str, max_new_tokens: int = 80) -> str:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    model_path = find_model_path()
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True).eval()

    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

def debug_model_output(prompt: str) -> str:
    return '{"tool":"get_current_price","args":{"symbol":"AAPL"}}'

def model_planner(messages: list[dict]) -> str:
    transcript = "\n".join(message["content"] for message in messages)
    prompt = (
        "Return JSON only for the next tool call.\n"
        'Available tool: {"tool":"get_current_price","args":{"symbol":"AAPL"}}\n'
        f"Transcript:\n{transcript}"
    )
    return ask_local_model(prompt) if USE_LOCAL_MODEL else debug_model_output(prompt)

messages = [{"role": "user", "content": "Can Alice add more AAPL?"}]
print(model_planner(messages))

## 4. Typed Tool Interfaces

Now expose three Chronos-style tools with Pydantic schemas. The schema is the runtime contract for untrusted model arguments.

In [ ]:
class PortfolioAllocationArgs(BaseModel):
    client_id: int

class GuidelineCheckArgs(BaseModel):
    symbol: str
    proposed_allocation_pct: float

def get_portfolio_allocation(client_id: int) -> dict:
    return {"client_id": client_id, "AAPL": 32.0, "cash": 18.0}

def check_guidelines(symbol: str, proposed_allocation_pct: float) -> dict:
    allowed = proposed_allocation_pct <= 35.0
    return {"symbol": symbol, "allowed": allowed, "limit_pct": 35.0}

TOOL_SCHEMAS = {
    "get_current_price": CurrentPriceArgs,
    "get_portfolio_allocation": PortfolioAllocationArgs,
    "check_guidelines": GuidelineCheckArgs,
}

TOOL_FUNCTIONS = {
    "get_current_price": get_current_price,
    "get_portfolio_allocation": get_portfolio_allocation,
    "check_guidelines": check_guidelines,
}

print(TOOL_SCHEMAS["get_current_price"].model_json_schema())

## 5. Bare-Metal Runtime Boundary

The runtime turns model output into controlled execution:

```text
parse JSON -> check tool registry -> validate args -> call Python function
```

In [ ]:
def execute_tool(raw_step: str) -> dict:
    step = json.loads(raw_step)
    name = step["tool"]

    if name not in TOOL_FUNCTIONS:
        raise ValueError(f"Unknown tool: {name}")

    args = TOOL_SCHEMAS[name].model_validate(step["args"])
    # Generic dispatch: Python calls the selected tool here.
    return TOOL_FUNCTIONS[name](**args.model_dump())

good = '{"tool":"get_current_price","args":{"symbol":"AAPL"}}'
print(execute_tool(good))

## 6. Error Observation

Bad model output should not crash the whole workflow. Convert failures into useful observations.

In [ ]:
def safe_execute(raw_step: str) -> dict:
    try:
        return {"ok": True, "result": execute_tool(raw_step)}
    except Exception as error:
        return {
            "ok": False,
            "error_type": type(error).__name__,
            "message": str(error),
        }

bad = '{"tool":"get_current_price","args":{"ticker":"AAPL"}}'
print(safe_execute(bad))

## 7. Complete Advisor Loop

This is the controller loop plus planner plus typed tool runtime. The planner is deterministic here so everyone can complete the lab path.

In [ ]:
def format_observation(tool: str, result: dict) -> dict:
    return {"role": "tool", "content": f"{tool} observation: {result}"}

def advisor_planner(messages: list[dict]) -> str:
    transcript = " ".join(message["content"] for message in messages)
    if "get_current_price observation" not in transcript:
        return '{"tool":"get_current_price","args":{"symbol":"AAPL"}}'
    if "get_portfolio_allocation observation" not in transcript:
        return '{"tool":"get_portfolio_allocation","args":{"client_id":1}}'
    if "check_guidelines observation" not in transcript:
        return '{"tool":"check_guidelines","args":{"symbol":"AAPL","proposed_allocation_pct":36}}'
    return '{"final":"Alice should not raise AAPL to 36%; the guideline limit is 35%."}'

messages = [{"role": "user", "content": "Can Alice add more AAPL under guidelines?"}]

for turn in range(5):
    raw_step = advisor_planner(messages)
    step = json.loads(raw_step)

    if "final" in step:
        print("FINAL:", step["final"])
        break

    outcome = safe_execute(raw_step)
    print("TURN", turn, "OUTCOME:", outcome)

    if outcome["ok"]:
        messages.append(format_observation(step["tool"], outcome["result"]))
    else:
        messages.append({"role": "tool", "content": f"ERROR observation: {outcome}"})

## 8. Add Telemetry

Telemetry makes the loop inspectable. Record raw model output, raw args, validated args, results, exceptions, and elapsed time.

In [ ]:
trace = []

def execute_with_trace(turn: int, raw_step: str) -> dict:
    started = perf_counter()
    record = {"turn": turn, "raw_model_output": raw_step}
    try:
        step = json.loads(raw_step)
        record["tool"] = step.get("tool")
        record["raw_args"] = step.get("args")

        args = TOOL_SCHEMAS[step["tool"]].model_validate(step["args"])
        record["validated_args"] = args.model_dump()

        # Traced dispatch: Python calls the selected tool here.
        result = TOOL_FUNCTIONS[step["tool"]](**args.model_dump())
        record["result"] = result
        return result
    except Exception as error:
        record["exception"] = type(error).__name__
        record["message"] = str(error)
        raise
    finally:
        record["elapsed_ms"] = round((perf_counter() - started) * 1000, 2)
        trace.append(record)

for turn, raw_step in enumerate([good, bad]):
    try:
        print("RESULT:", execute_with_trace(turn, raw_step))
    except Exception:
        print("ERROR RECORDED")

for record in trace:
    print(json.dumps(record))

## 9. Final Agent Run With Trace

Run the complete advisor loop again, this time with telemetry for every tool call.

In [ ]:
trace = []
messages = [{"role": "user", "content": "Can Alice add more AAPL under guidelines? Show the facts you checked."}]
final_answer = None

for turn in range(5):
    raw_step = advisor_planner(messages)
    step = json.loads(raw_step)

    if "final" in step:
        final_answer = step["final"]
        break

    try:
        result = execute_with_trace(turn, raw_step)
        messages.append(format_observation(step["tool"], result))
    except Exception as error:
        messages.append({"role": "tool", "content": f"ERROR observation: {type(error).__name__}: {error}"})

print("FINAL ANSWER:", final_answer)
print("\nFACTS CHECKED:")
for message in messages:
    if message["role"] == "tool":
        print("-", message["content"])

print("\nTRACE JSONL:")
for record in trace:
    print(json.dumps(record))

## 10. Stretch

After the deterministic loop works:

- replace `advisor_planner()` with a local model planner;
- save `trace` as JSONL;
- connect `get_current_price()` to Chronos simulated-date prices;
- connect `get_portfolio_allocation()` to a portfolio snapshot;
- keep deterministic Python responsible for guideline checks.

Framework agent loops package these same parts: planner, tool registry, runtime, state, trace, and loop limits.